# ml_tool 回归模型使用示例

完整流程：数据准备 → 特征分析 → 特征筛选 → XGBoost 回归训练 → 回归评估报告输出

In [ ]:
import pandas as pd
import numpy as np
from ml_tool import FeatureAnalyzer, FeatureSelector, ModelTrainer, ReportGenerator

# 生成示例回归数据（替换为你的实际数据）
# 目标变量 y 为连续值，例如贷款金额、逾期天数、收入预测等
np.random.seed(42)
n = 3000
f1 = np.random.randn(n)
f2 = np.random.rand(n) * 10
f3 = np.random.randn(n) * 2
f4 = np.random.rand(n) * 5
f5 = np.random.randn(n)

# 目标变量与特征有一定线性关系，加上噪声
y = 2.5 * f1 - 1.2 * f2 + 0.8 * f3 + np.random.randn(n) * 3

df = pd.DataFrame({
    'f1': f1, 'f2': f2, 'f3': f3, 'f4': f4, 'f5': f5,
    'y': y,
    'dataset': np.random.choice(['train', 'test', 'oot'], n, p=[0.6, 0.2, 0.2])
})
# 加入部分缺失
df.loc[df.sample(frac=0.04).index, 'f2'] = np.nan
df.loc[df.sample(frac=0.02).index, 'f4'] = np.nan

print('数据形状:', df.shape)
print('数据集划分:', df['dataset'].value_counts().to_dict())
print('目标变量统计:')
print(df['y'].describe().round(3))

## 1. 特征分析

In [ ]:
feature_cols = ['f1', 'f2', 'f3', 'f4', 'f5']
analyzer = FeatureAnalyzer(df[feature_cols])

print('=== 缺失率 ===')
display(analyzer.missing_rate())

print('=== 一值率 ===')
display(analyzer.single_value_rate())

print('=== 分位数统计 ===')
display(analyzer.quantile_stats())

In [ ]:
# PSI：以 train 为基准组，oot 为对照组
base_df    = df[df['dataset'] == 'train'][feature_cols]
compare_df = df[df['dataset'] == 'oot'][feature_cols]

psi_df = analyzer.psi(base_df, compare_df)
print('=== PSI（train vs OOT）===')
display(psi_df)

print('=== 特征相关性（|r|≥0.2）===')
display(analyzer.correlation(threshold=0.2))

## 2. 特征筛选

回归任务没有 WOE/IV，跳过分箱步骤，直接按缺失率、PSI、相关性筛选。

IV 阈值设为 0（不按 IV 过滤），其余阈值保持默认。

In [ ]:
train_df = df[df['dataset'] == 'train'].reset_index(drop=True)

selector = FeatureSelector(
    iv_threshold=0.0,     # 回归不用 IV 筛选，设为 0 关闭
    psi_threshold=0.5,
    missing_threshold=0.98,
    corr_threshold=0.97,
)
selector.fit(
    df=train_df[feature_cols + ['y']],
    target='y',
    base_df=base_df,
    compare_df=compare_df,
)

print('保留特征:', selector.get_selected())
print('剔除特征:', selector.dropped_cols)
display(selector.get_report())

In [ ]:
# 输出特征筛选 Excel 报告
reporter = ReportGenerator(output_dir='./reports')
analysis = {
    '缺失率': analyzer.missing_rate(),
    '分位数统计': analyzer.quantile_stats(),
    'PSI': psi_df,
    '相关性': analyzer.correlation(threshold=0.0),
}
path = reporter.feature_selection_report(selector.get_report(), analysis_results=analysis,
                                          filename='回归_特征筛选报告')
print('特征筛选报告已保存：', path)

## 3. XGBoost 回归模型训练

两轮 Hyperopt 调参流程：
- 第一轮：全特征宽搜索 → 训出初始模型 → 按 gain 重要性剔除无贡献特征
- 第二轮：精简特征集上细搜索 → 训出最终模型

优化目标：`OOT_RMSE + |TR_RMSE - TS_RMSE|`（最小化 OOT 误差，同时抑制过拟合）

In [ ]:
selected = selector.get_selected()

X_train = df[df['dataset'] == 'train'][selected]
y_train = df[df['dataset'] == 'train']['y']
X_test  = df[df['dataset'] == 'test'][selected]
y_test  = df[df['dataset'] == 'test']['y']
X_oot   = df[df['dataset'] == 'oot'][selected]
y_oot   = df[df['dataset'] == 'oot']['y']

# n_trials 设小以便快速演示，实际建议 100
trainer = ModelTrainer(model_type='xgboost_reg', n_trials=20)
trainer.fit(X_train, y_train, X_test, y_test, X_oot, y_oot)

print('第一轮筛选后入模特征数:', len(trainer.selected_features))
print('入模特征列表:', trainer.selected_features)
print('\n最优超参数:')
for k, v in trainer._trainer.best_params.items():
    print(f'  {k}: {v}')

In [ ]:
# 查看调参过程：每次 trial 的 train/test/oot RMSE 和 loss
print('=== 第一轮调参过程（前10次 trial）===')
display(trainer.trials_log.head(10))

print('\n最优 trial 指标:')
best_row = trainer.trials_log.loc[trainer.trials_log['loss'].idxmin()]
display(best_row.to_frame().T)

## 4. 回归模型评估报告

指标：RMSE、MAE、R²、Pearson/Spearman 相关系数、分桶分析

输出：Excel（多 sheet）+ HTML 报告

In [ ]:
# 获取三个数据集的预测值
pred_train = trainer.predict(X_train)
pred_test  = trainer.predict(X_test)
pred_oot   = trainer.predict(X_oot)

datasets_reg = {
    '训练集': (y_train.values, pred_train),
    '测试集': (y_test.values,  pred_test),
    'OOT':   (y_oot.values,   pred_oot),
}

reg_result = reporter.regression_report(
    datasets_reg,
    filename='XGBoost_回归模型评估报告',
    n_bins=10
)

print('=== 汇总指标 ===')
display(reg_result['summary'])
print('\nExcel 报告:', reg_result['excel_path'])
print('HTML  报告:', reg_result['html_path'])

In [ ]:
# 查看 OOT 分桶分析：每桶的预测均值 vs 真实均值
print('=== OOT 分桶分析（10桶）===')
display(reg_result['bucket_tables']['OOT_分桶分析'])

In [ ]:
# 预测值 vs 真实值散点图（可选，需要 matplotlib）
try:
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, (name, (y_true, y_pred)) in zip(axes, datasets_reg.items()):
        ax.scatter(y_true, y_pred, alpha=0.3, s=10)
        mn, mx = min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())
        ax.plot([mn, mx], [mn, mx], 'r--', lw=1.5, label='理想线')
        ax.set_xlabel('真实值')
        ax.set_ylabel('预测值')
        ax.set_title(name)
        ax.legend()
    plt.tight_layout()
    plt.savefig('./reports/预测值_vs_真实值.png', dpi=120)
    plt.show()
    print('散点图已保存：./reports/预测值_vs_真实值.png')
except ImportError:
    print('matplotlib 未安装，跳过散点图')